# Evaluate Essays - Interactive Scoring & Feedback

This notebook enables interactive scoring of new essays using trained ASAP-AES models.

**Capabilities:**
- Load trained XGBoost models from disk
- Score individual essays in real-time
- Generate personalized improvement feedback
- Display feature-based analysis (which features drove the score)
- Batch evaluate multiple essays from CSV
- Export results with confidence scores

**Prerequisites:**
- Completion of Train_Essay_Scoring_Model.ipynb
- Trained models in `../models/` directory
- Python packages: pandas, numpy, scikit-learn, xgboost, spacy, textblob

In [ ]:
# Setup and Imports
# =================

import os
import sys
import json
import pickle
import warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Add src to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

# Import custom modules
try:
    from feature_extractor import EssayFeatureExtractor
    from essay_evaluator import EssayEvaluator
    from feedback_generator import FeedbackGenerator
    from metrics import compute_qwk
    print("✓ All custom modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")

# Configuration
MODEL_DIR = Path('../models')
OUTPUT_DIR = Path('../outputs')
CONFIG_FILE = Path('../configs/essay_scoring_config.json')

# Create directories if needed
OUTPUT_DIR.mkdir(exist_ok=True)

# Load configuration
try:
    with open(CONFIG_FILE) as f:
        config = json.load(f)
    print(f"✓ Configuration loaded from {CONFIG_FILE.name}")
except FileNotFoundError:
    print(f"⚠️  Configuration file not found: {CONFIG_FILE}")
    config = {}

# Suppress warnings
warnings.filterwarnings('ignore')

print(f"✓ Environment initialized")
print(f"  Model directory: {MODEL_DIR.absolute()}")
print(f"  Output directory: {OUTPUT_DIR.absolute()}")

In [ ]:
# Step 1: Load Trained Models
# ===========================

print("=" * 80)
print("LOADING TRAINED MODELS")
print("=" * 80)

# Initialize feature extractor
extractor = EssayFeatureExtractor()

# Load trained models
loaded_models = {}
model_files = list(MODEL_DIR.glob('essay_set_*_model.pkl'))

if not model_files:
    print("❌ No trained models found in models directory")
    print(f"   Expected location: {MODEL_DIR.absolute()}")
    print("   Please run Train_Essay_Scoring_Model.ipynb first")
else:
    for model_file in sorted(model_files):
        try:
            with open(model_file, 'rb') as f:
                model = pickle.load(f)
            # Extract essay set ID from filename (essay_set_1_model.pkl -> 1)
            set_id = int(model_file.stem.split('_')[2])
            loaded_models[set_id] = model
            print(f"✓ Loaded model for Essay Set {set_id}")
        except Exception as e:
            print(f"❌ Failed to load {model_file.name}: {e}")
    
    print(f"\n✓ Total models loaded: {len(loaded_models)}")
    print(f"  Available essay sets: {sorted(loaded_models.keys())}")

In [ ]:
# Step 2: Interactive Essay Scoring
# ==================================\n\nprint(\"\\n\" + \"=\" * 80)\nprint(\"INTERACTIVE ESSAY EVALUATION\")\nprint(\"=\" * 80)\n\n# Define helper function to score an essay\ndef score_essay(essay_text, essay_set_id):\n    \"\"\"\n    Score a single essay using the trained model.\n    \n    Args:\n        essay_text (str): The essay text to score\n        essay_set_id (int): The essay set ID (1-8)\n    \n    Returns:\n        dict: Scoring results with prediction, confidence, and feedback\n    \"\"\"\n    if essay_set_id not in loaded_models:\n        return {\"error\": f\"No model for essay set {essay_set_id}. Available: {sorted(loaded_models.keys())}\"}\n    \n    if not essay_text or len(essay_text.strip()) < 10:\n        return {\"error\": \"Essay too short (minimum 10 characters)\"}\n    \n    try:\n        # Extract features\n        features = extractor.extract_features(essay_text)\n        feature_vector = np.array([list(features.values())])\n        \n        # Predict score\n        model = loaded_models[essay_set_id]\n        predicted_score = model.predict(feature_vector)[0]\n        \n        # Round to nearest integer\n        rounded_score = round(predicted_score)\n        \n        return {\n            \"essay_set\": essay_set_id,\n            \"predicted_score\": round(predicted_score, 2),\n            \"rounded_score\": int(rounded_score),\n            \"confidence\": \"high\" if predicted_score == rounded_score else \"medium\",\n            \"features\": features,\n            \"essay_length\": len(essay_text.split())\n        }\n    except Exception as e:\n        return {\"error\": f\"Scoring failed: {str(e)}\"}\n\n# Example: Score an essay\nprint(\"\\n📝 Example: Score an essay\")\nprint(\"\\nEdit the essay_text and essay_set_id below, then run this cell:\")\nprint()\n\n# USER INPUT SECTION\nessay_text = \"\"\"\nTechnology has revolutionized education. Students now have access to vast resources \nonline. Teachers can use interactive tools to engage learners. However, excessive \nscreen time may reduce face-to-face interaction. Overall, technology provides \nenormous benefits when used responsibly.\n\"\"\"\n\nessay_set_id = 1  # Change this to 1-8 based on your needs\n\n# Score the essay\nif loaded_models:\n    result = score_essay(essay_text, essay_set_id)\n    \n    if \"error\" in result:\n        print(f\"❌ {result['error']}\")\n    else:\n        print(f\"✓ Essay Set {result['essay_set']} Evaluation:\")\n        print(f\"\\n  Predicted Score: {result['predicted_score']}\")\n        print(f\"  Rounded Score: {result['rounded_score']}\")\n        print(f\"  Confidence: {result['confidence']}\")\n        print(f\"  Essay Length: {result['essay_length']} words\")\n        print(f\"\\n  Key Features:\")\n        for feature, value in list(result['features'].items())[:5]:\n            print(f\"    - {feature}: {value:.2f}\")\nelse:\n    print(\"❌ No models loaded. Cannot score essay.\")

In [ ]:
# Step 3: Generate Personalized Feedback
# ======================================

print("\n" + "=" * 80)
print("GENERATING FEEDBACK")
print("=" * 80)

def generate_essay_feedback(features_dict, predicted_score):
    """
    Generate human-readable feedback based on essay features.
    
    Args:
        features_dict (dict): Feature values extracted from essay
        predicted_score (float): The predicted score
    
    Returns:
        dict: Feedback with strengths, weaknesses, and suggestions
    """
    feedback = {
        "strengths": [],
        "weaknesses": [],
        "suggestions": []
    }
    
    # Analyze features
    word_count = features_dict.get('word_count', 0)
    avg_sentence_length = features_dict.get('avg_sentence_length', 0)
    unique_word_ratio = features_dict.get('unique_word_ratio', 0)
    spelling_errors = features_dict.get('spelling_error_rate', 0)
    
    # Generate feedback based on features
    if word_count > 200:
        feedback['strengths'].append("✓ Good essay length shows thorough response")
    elif word_count < 50:
        feedback['weaknesses'].append("✗ Essay is too short to adequately address the prompt")
        feedback['suggestions'].append("→ Expand your response with more specific examples and details")
    
    if avg_sentence_length > 12 and avg_sentence_length < 20:
        feedback['strengths'].append("✓ Sentence length is well-balanced")
    elif avg_sentence_length > 25:\n        feedback['weaknesses'].append("✗ Sentences are too long and complex")\n        feedback['suggestions'].append("→ Break down long sentences for clarity")\n    \n    if unique_word_ratio > 0.7:\n        feedback['strengths'].append("✓ Good vocabulary variety demonstrates diverse language use")\n    else:\n        feedback['weaknesses'].append("✗ Limited vocabulary variation")\n        feedback['suggestions'].append("→ Use synonyms and vary word choice")\n    \n    if spelling_errors > 0.05:\n        feedback['weaknesses'].append(f"✗ Multiple spelling errors detected ({spelling_errors*100:.1f}%)")\n        feedback['suggestions'].append("→ Proofread carefully before submitting")\n    else:\n        feedback['strengths'].append("✓ Spelling and mechanics are accurate")\n    \n    return feedback\n\n# Generate feedback for the example essay\nif 'result' in locals() and 'error' not in result:\n    feedback = generate_essay_feedback(result['features'], result['predicted_score'])\n    \n    print(f"\\n📋 Feedback for Essay Set {result['essay_set']}:\")\n    print(f\"   Score: {result['predicted_score']}/12\")\n    print(f\"\\n💪 Strengths:\")\n    for strength in feedback['strengths']:\n        print(f\"   {strength}\")\n    \n    print(f\"\\n⚠️  Areas for Improvement:\")\n    for weakness in feedback['weaknesses']:\n        print(f\"   {weakness}\")\n    \n    print(f\"\\n💡 Suggestions:\")\n    for suggestion in feedback['suggestions']:\n        print(f\"   {suggestion}\")\nelse:\n    print("⚠️  Run the previous cell to generate feedback.")

In [ ]:
# Step 4: Batch Evaluation (Optional)
# ===================================

print("\n" + "=" * 80)
print("BATCH EVALUATION")
print("=" * 80)

def batch_evaluate_essays(csv_file, essay_col, set_col, output_file=None):
    \"\"\"\n    Evaluate multiple essays from a CSV file.\n    \n    Expected CSV columns: essay_text, essay_set, [score (optional)]\n    \"\"\"\n    try:\n        # Load CSV\n        df = pd.read_csv(csv_file)\n        print(f"✓ Loaded {len(df)} essays from {csv_file.name}")\n        \n        results = []\n        \n        for idx, row in df.iterrows():\n            essay_text = row[essay_col]\n            essay_set = int(row[set_col])\n            \n            result = score_essay(essay_text, essay_set)\n            \n            if 'error' not in result:\n                result_row = {\n                    'essay_id': idx,\n                    'essay_set': essay_set,\n                    'predicted_score': result['predicted_score'],\n                    'rounded_score': result['rounded_score'],\n                    'confidence': result['confidence'],\n                    'essay_length': result['essay_length']\n                }\n                \n                # Add actual score if available\n                if 'score' in row or 'actual_score' in df.columns:\n                    score_col = 'score' if 'score' in row else 'actual_score'\n                    result_row['actual_score'] = row[score_col]\n                \n                results.append(result_row)\n            \n            if (idx + 1) % 10 == 0:\n                print(f\"  Processed {idx + 1} essays...\")\n        \n        # Create results dataframe\n        results_df = pd.DataFrame(results)\n        \n        # Save results\n        if output_file:\n            results_df.to_csv(output_file, index=False)\n            print(f"\\n✓ Results saved to {output_file}\")\n        \n        # Display summary\n        print(f\"\\n📊 Batch Evaluation Summary:\")\n        print(f\"  Evaluated: {len(results)} essays\")\n        print(f\"  Average predicted score: {results_df['predicted_score'].mean():.2f}\")\n        print(f\"  Confidence distribution:\\n{results_df['confidence'].value_counts()}\")\n        \n        return results_df\n        \n    except Exception as e:\n        print(f\"❌ Batch evaluation failed: {e}\")\n        return None\n\nprint(\"\\n📌 To batch evaluate essays:\")\nprint(\"\\n1. Prepare a CSV file with columns: essay_text, essay_set\")\nprint(\"2. Run this cell with the CSV file path:\")\nprint()\nprint(\"   results_df = batch_evaluate_essays(\")\nprint(\"       csv_file=Path('../data/essays_to_score.csv'),\")\nprint(\"       essay_col='essay_text',\")\nprint(\"       set_col='essay_set',\")\nprint(\"       output_file=Path('../outputs/batch_scores.csv')\")\nprint(\"   )\")\nprint()\nprint(\"3. Results will be saved to ../outputs/batch_scores.csv\")

In [ ]:
# Summary & API Integration
# ==========================

print("\n" + "=" * 80)
print("READY FOR DEPLOYMENT")
print("=" * 80)

if loaded_models:
    print(f"\n✓ {len(loaded_models)} models loaded and ready")
    print(f"✓ Evaluation functions defined")
    print(f"✓ Feedback generation enabled")
    
    print("\n" + "=" * 80)
    print("DEPLOYMENT OPTIONS:")
    print("=" * 80)
    
    print("\n1️⃣  NOTEBOOK USAGE (Current)")
    print("   - Edit essay_text and essay_set_id in Step 2")
    print("   - Run cells to score essays interactively")
    print("   - Use batch_evaluate_essays() for multiple essays")
    
    print("\n2️⃣  COMMAND-LINE USAGE")
    print("   - Run: python evaluate.py --data essays.csv --essay-set 1")
    print("   - Scores will be saved to outputs/")
    
    print("\n3️⃣  FLASK API (Phase 2)")
    print("   - Endpoint: POST /evaluate")
    print("   - Input: {'essay_text': '...', 'essay_set_id': 1}")
    print("   - Output: {'score': 8.5, 'feedback': {...}, 'confidence': 'high'}")
    
    print("\n" + "=" * 80)
    print("NEXT STEPS:")
    print("=" * 80)
    print("\n✓ Score essays using the interactive cells above")
    print("✓ Review feedback and scores")
    print("✓ Batch evaluate large datasets")
    print("✓ Export results to CSV for analysis")
    print("✓ Deploy Flask API for integration with Flutter app (Phase 2)")
    
    print(f"\n✓ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("\n⚠️  No models loaded. Please:")
    print("   1. Check that Train_Essay_Scoring_Model.ipynb has been run")
    print("   2. Verify trained models exist in ../models/")
    print("   3. Re-run the model loading cell above")

print("\n" + "=" * 80)